# Proyecto P03: Segmentación de clientes — Guía del ejercicio

En este proyecto vas a practicar la segmentación, uno de los usos más importantes del aprendizaje no supervisado: calcularás los indicadores **RFM** (*Recency*, *Frequency*, *Monetary*) de cada cliente de una tienda en línea, los agruparás con **k-Means** y combinarás los grupos en una calificación que asigne a cada cliente un segmento de valor.

Las instrucciones originales del proyecto están en [instrucciones.md](instrucciones.md). Cuando termines, compara tu trabajo con la solución: [p03-segmentacion-de-clientes-solucion.ipynb](p03-segmentacion-de-clientes-solucion.ipynb).

## Contexto

Todas las empresas funcionan con base en clientes, y no todos valen lo mismo. Una tienda en línea con sede en el Reino Unido, que vende artículos de regalo a clientes de 38 países, quiere dirigir sus campañas según el comportamiento de compra de cada cliente. Con los datos de sus clientes podría hacer estadística descriptiva, predecir el abandono o estimar el valor del cliente a lo largo del tiempo (CLTV); en este proyecto te pide **segmentar a los clientes** con la metodología RFM:

- **Recency:** qué tan reciente es la última compra del cliente.
- **Frequency:** con qué frecuencia compra.
- **Monetary:** cuánto ha gastado en total.

## Datos

El archivo `online-retail.csv` está en esta misma carpeta. Tiene 541 909 filas, una por línea de factura, entre el 1 de diciembre de 2019 y el 9 de diciembre de 2020. Pesa unos 46 MB y está codificado en **ISO-8859-1** (`encoding="ISO-8859-1"`).

| Columna | Descripción |
| --- | --- |
| `INVOICE_NO` | Número de factura; las que empiezan con `C` son cancelaciones |
| `STOCK_CODE` | Código del producto |
| `DESCRIPTION` | Descripción del producto (con faltantes) |
| `QUANTITY` | Unidades de la línea; negativa en devoluciones |
| `INVOICE_DATE` | Fecha y hora, en texto con formato día/mes/año (`01/12/2019 08:26`) |
| `UNIT_PRICE` | Precio unitario en libras esterlinas |
| `CUSTOMER_ID` | Identificador del cliente; falta en cerca de una cuarta parte de las filas |
| `REGION` | País del cliente |

## Instrucciones

- Trabaja en este notebook, ejecutándolo desde su carpeta (`uv run jupyter lab` desde la raíz del repositorio). También puedes usar Visual Studio Code o Google Colab.
- Puedes usar las librerías de tu preferencia, aunque se recomiendan pandas, NumPy, seaborn, matplotlib y scikit-learn.
- Escribe tu código en las celdas vacías que siguen a cada tarea y acompaña la entrega con un breve resumen de tus conclusiones.
- Tiempo estimado: 60 minutos.

La celda siguiente importa las librerías, carga los datos y define `order_cluster`, una función auxiliar que reordena las etiquetas de k-Means para que sigan el orden de la media de un indicador (k-Means numera los grupos al azar).

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("online-retail.csv", encoding="ISO-8859-1")


# Reordena los clusters de `cluster_field_name` según la media de `target_field_name`:
# el cluster 0 es el de menor media si ascending=True, o el de mayor media si ascending=False.
def order_cluster(cluster_field_name, target_field_name, df, ascending):
    new_cluster_field_name = "new_" + cluster_field_name
    df_new = df.groupby(cluster_field_name)[target_field_name].mean().reset_index()
    df_new = df_new.sort_values(by=target_field_name, ascending=ascending).reset_index(drop=True)
    df_new["index"] = df_new.index
    df_final = pd.merge(df, df_new[[cluster_field_name, "index"]], on=cluster_field_name)
    df_final = df_final.drop([cluster_field_name], axis=1)
    df_final = df_final.rename(columns={"index": cluster_field_name})
    return df_final


df.head()

## Tarea 1. Explorar y limpiar los datos

1. Revisa la estructura con `info()` y convierte `INVOICE_DATE` a fecha (formato `%d/%m/%Y %H:%M`).
2. Calcula el porcentaje de valores nulos de `CUSTOMER_ID`. Rellénalos con `-1`, convierte la columna a entero y después, junto con `INVOICE_NO`, `STOCK_CODE` y `REGION`, a tipo `category`. Convierte `DESCRIPTION` a texto.
3. Revisa la estadística descriptiva de las columnas numéricas y categóricas. Examina las filas con `QUANTITY` o `UNIT_PRICE` negativos y decide, justificándolo, si las conservas.
4. Elimina los registros sin cliente (`CUSTOMER_ID == -1`), porque no se pueden agrupar por cliente.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 2. Recency

Indicador que dice qué tan reciente es la compra de un cliente.

1. Crea un DataFrame `customer` con un registro por cliente único (por ejemplo, con el número de líneas de factura de cada uno). Al agrupar por columnas de tipo `category`, usa `observed=True`.
2. Obtén la fecha de la última compra de cada cliente.
3. Calcula `RECENCY` como los días entre la fecha de compra más reciente de todo el conjunto y la última compra de cada cliente, y únelo a `customer`.
4. Grafica un histograma de `RECENCY` e imprime su estadística de resumen.

In [ ]:
# Tu código aquí

## Tarea 3. Frequency

Frecuencia con la que un cliente compra uno o más productos.

1. Calcula `FREQUENCY` como el número de facturas distintas de cada cliente y únelo a `customer`.
2. Grafica un histograma de `FREQUENCY` e imprime su estadística de resumen.

In [ ]:
# Tu código aquí

## Tarea 4. Monetary

Monto total que ha gastado un cliente en tus productos.

1. Calcula el monto de cada línea de factura (`QUANTITY` × `UNIT_PRICE`).
2. Suma los montos por cliente para obtener `MONETARY` y únelo a `customer`.
3. Grafica un histograma de `MONETARY` e imprime su estadística de resumen.

In [ ]:
# Tu código aquí

## Tarea 5. Número de clusters: método del codo

¿Cuál es el número óptimo de clusters? Tomando `RECENCY` como referencia, entrena k-Means para $k = 1, \dots, 9$ (`max_iter=1000`), guarda la inercia de cada modelo y construye la gráfica de codo.

In [ ]:
# Tu código aquí

## Tarea 6. Clusters de k-Means para cada indicador

Con el número de clusters que sugiere el codo (4), y para cada indicador por separado:

1. Entrena k-Means sobre el indicador y guarda las etiquetas en `RECENCY_CLUSTER`, `FREQUENCY_CLUSTER` y `MONETARY_CLUSTER`.
2. Ordena los clusters con `order_cluster` de modo que un número **mayor** signifique un cliente **mejor**. Piensa en qué sentido debes ordenar cada uno: una recencia alta es mala, mientras que una frecuencia o un monto altos son buenos.
3. Muestra la estadística descriptiva del indicador en cada cluster.

In [ ]:
# Tu código aquí

## Tarea 7. Score de segmentación

k-Means da una segmentación general, pero puedes personalizarla con una métrica que califique el valor de cada cliente.

1. Crea `SCORE` como la suma de los tres clusters y calcula la media de `RECENCY`, `FREQUENCY` y `MONETARY` para cada valor del score.
2. Escribe una función que asigne el segmento según el score: `"Low-Value"` si es menor o igual a 1; `"Average"` si es mayor que 1 y menor o igual a 4; `"Potential"` si es mayor que 4 y menor o igual a 6, y `"High-Value"` si es mayor que 6. Aplícala para crear la columna `SEGMENT`.
3. Revisa la tabla final e imprime la proporción de clientes en cada segmento.

In [ ]:
# Tu código aquí

## Tarea 8. Visualizar e interpretar los segmentos

1. Grafica la dispersión de `MONETARY` frente a `RECENCY`, con un color por segmento (el material original sugiere el estilo `"bmh"`).
2. Grafica la dispersión de `MONETARY` frente a `FREQUENCY`, también por segmento.
3. Interpreta cada gráfica y cierra con un breve resumen de tus conclusiones.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Antes de terminar

- [ ] El notebook se ejecuta de principio a fin sin errores.
- [ ] Cada cliente tiene `RECENCY`, `FREQUENCY` y `MONETARY`.
- [ ] Los clusters están ordenados para que un número mayor signifique un mejor cliente.
- [ ] Cada cliente tiene un `SCORE` y un `SEGMENT`.
- [ ] Interpretaste las gráficas y escribiste tus conclusiones.

Cuando termines, compara tu trabajo con la solución: [p03-segmentacion-de-clientes-solucion.ipynb](p03-segmentacion-de-clientes-solucion.ipynb).